# 01 · Comic Text Bubble Detection

This notebook fetches a pretrained RT-DETR model from Hugging Face, lets you upload a comic page, and highlights detected speech bubbles that contain text.

## 1. Init
Download (cached after first run) and load the `ogkalu/comic-text-and-bubble-detector` model. The model targets the `text_bubble` class (index 1).

In [1]:
from pathlib import Path
import io

import torch
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display

MODEL_ID = "ogkalu/comic-text-and-bubble-detector"
CACHE_DIR = Path.home() / ".cache" / "comic-bubble-detector"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

processor = AutoImageProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CLASS_LABELS = {0: "bubble", 1: "text_bubble", 2: "text_free"}
TARGET_CLASS = "text_bubble"
TARGET_ID = 1
print("Done!")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Done!


## 2. Upload an image
Use the file picker to provide a JPG or PNG comic page, then click **Continue** to run detection.

In [2]:
import ipywidgets as widgets
from IPython.display import display, Image

upload = widgets.FileUpload(
    accept='image/png, image/jpeg',
    multiple=False,
    description='Upload page'
)
score_threshold = widgets.FloatSlider(
    value=0.3, min=0.1, max=0.9, step=0.05, description='Score ≥'
)

display(widgets.HBox([upload, score_threshold]))
print(upload)


FileUpload(value=(), accept='image/png, image/jpeg', description='Upload page')


In [3]:
# Run bubble recognition
import PIL.Image

results_out2 = widgets.Output()
FINAL_OUTPUTS = {}

def load_image_from_upload(widget: widgets.FileUpload) -> PIL.Image:
    if not widget.value:
        raise ValueError("Upload an image before running detection.")

    file_info = widget.value[0]   # not next(iter(...))
    return PIL.Image.open(io.BytesIO(bytes(file_info["content"]))).convert("RGB")


def draw_boxes(base_image: Image.Image, boxes, scores):
    annotated = base_image.copy()
    draw = ImageDraw.Draw(annotated, 'RGBA')
    for box, score in zip(boxes, scores):
        x0, y0, x1, y1 = [float(v) for v in box]
        draw.rectangle([(x0, y0), (x1, y1)], outline=(255, 64, 64, 255), width=3)
    return annotated


def process():
    print("processing..")
    results_out2.clear_output()
    try:
        image = load_image_from_upload(upload)
    except ValueError as exc:
        with results_out2:
            print(exc)
        return

    inputs = processor(images=image, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]], device=device)
    detections = processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=score_threshold.value
    )[0]

    boxes = detections['boxes']
    scores = detections['scores']
    labels = detections['labels']

    mask = labels == TARGET_ID
    boxes = boxes[mask].cpu().numpy()
    scores = scores[mask].cpu().numpy()

    annotated = draw_boxes(image, boxes, scores)

    FINAL_OUTPUTS['image'] = annotated
    FINAL_OUTPUTS['boxes'] = boxes
    FINAL_OUTPUTS['scores']=scores

    with results_out2:
        display(annotated)
        if len(boxes) == 0:
            print('No text_bubble regions detected above the current score threshold.')
        else:
            print(f'Detected {len(boxes)} text_bubble regions:')
            for idx, (box, score) in enumerate(zip(boxes, scores), start=1):
                x0, y0, x1, y1 = box.tolist()
                print(f"#{idx}: score={score:.2f}, box=[{x0:.1f}, {y0:.1f}, {x1:.1f}, {y1:.1f}]")

process()


display(results_out2)

processing..


Output()

## Save results for the next step

In [4]:
# Paths inside the repository where we cache intermediate results
from pathlib import Path
import json

ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATH = ARTIFACT_DIR / 'latest_text_bubble_detection.png'
BOXES_PATH = ARTIFACT_DIR / 'latest_text_bubble_boxes.json'

def save_latest_results(image, boxes, scores):
    image.save(IMAGE_PATH, format='PNG')
    payload = [
        {
            'score': float(score),
            'box': [float(v) for v in box]
        }
        for box, score in zip(boxes, scores)
    ]
    BOXES_PATH.write_text(json.dumps(payload, indent=2))
    return IMAGE_PATH, BOXES_PATH

# Invoke this helper after running detection
try:
    image_path, boxes_path = save_latest_results(FINAL_OUTPUTS['image'], FINAL_OUTPUTS['boxes'], FINAL_OUTPUTS['scores'])
    print(f'Saved annotated image to {image_path}')
    print(f'Saved box coordinates to {boxes_path}')
except NameError as err:
    print(err)
    print('Run the detection cell above first to generate results.')

Saved annotated image to artifacts/latest_text_bubble_detection.png
Saved box coordinates to artifacts/latest_text_bubble_boxes.json
